# AI/ML Model Optimization & High-Performance Storage Guide

This interactive Jupyter Notebook demonstrates best practices for optimizing, storing, and serving deep learning models (e.g. YOLOv8) for intelligent traffic monitoring.

### Table of Contents:
1. **Step 1: Dedicated Directory Structure & Git Safety**
2. **Step 2: Export to Optimized Runtime Formats (ONNX, OpenVINO, TensorRT)**
3. **Step 3: Precision Quantization (FP32 -> FP16 & INT8)**
4. **Step 4: High-Speed Storage & Model Cold-Start Loading**
5. **Step 5: Production Singleton Model Loader & Warm-Up Pass**
6. **Step 6: Live Inference Latency & FPS Benchmark**

In [ ]:
# --------------------------------------------------------------------------
# Step 0: Environment Setup & Verifications
# --------------------------------------------------------------------------
import os
import sys
import time
import shutil
from pathlib import Path
import numpy as np
import torch
from ultralytics import YOLO

print(f"Python Version:  {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU:      {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU mode.")


---
## Step 1: Organize in a Dedicated Directory & Exclude from Git

* **Rule**: Large weight binaries (`.pt`, `.onnx`, `.engine`) should never be stored in Git.
* Store models in dedicated folders like `models/weights/`, `models/onnx/`, `models/tensorrt/`.

In [ ]:
# Create dedicated directories for various model runtimes
PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd().resolve()
MODELS_DIR = PROJECT_ROOT / "models"
WEIGHTS_DIR = MODELS_DIR / "weights"
ONNX_DIR = MODELS_DIR / "onnx"
TRT_DIR = MODELS_DIR / "tensorrt"

for d in [WEIGHTS_DIR, ONNX_DIR, TRT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"[Ready] Directory: {d.relative_to(PROJECT_ROOT)}")

# Verify .gitignore excludes model weight files
gitignore_file = PROJECT_ROOT / ".gitignore"
if gitignore_file.exists():
    with open(gitignore_file, "r", encoding="utf-8") as f:
        content = f.read()
    has_pt = "*.pt" in content
    print(f"\.gitignore safeguards *.pt files: {has_pt}")
else:
    print("Warning: .gitignore file not found in project root.")


---
## Step 2: Export Model to High-Performance Runtimes

Standard PyTorch models execute through Python wrappers. Exporting to dedicated runtimes achieves higher throughput:
* **ONNX (`.onnx`)**: Standard open neural network exchange format; runs on ONNX Runtime (CPU/DirectML/CUDA).
* **OpenVINO**: Highly optimized for Intel CPUs, integrated GPUs, and VPUs.
* **TensorRT (`.engine`)**: Highest possible inference performance on NVIDIA GPUs with operator fusion.

In [ ]:
# Locate or download YOLOv8 baseline model
pt_model_path = PROJECT_ROOT / "yolov8n.pt"
if not pt_model_path.exists():
    print("Downloading yolov8n.pt baseline...")
    model = YOLO("yolov8n.pt")
else:
    print(f"Loading local model from {pt_model_path}")
    model = YOLO(str(pt_model_path))

# Export to ONNX with graph simplification
print("\n--- Exporting to ONNX ---")
exported_onnx_path = model.export(format="onnx", dynamic=False, simplify=True)
print(f"Exported ONNX model to: {exported_onnx_path}")

# Move exported file to the dedicated models/onnx folder
exported_path = Path(exported_onnx_path)
dest_onnx_path = ONNX_DIR / exported_path.name
if exported_path.exists() and exported_path != dest_onnx_path:
    shutil.move(str(exported_path), str(dest_onnx_path))
    print(f"Organized into: {dest_onnx_path}")


---
## Step 3: Apply Precision Quantization (FP32 -> FP16 & INT8)

* **FP32**: Default single-precision (32-bit floats).
* **FP16**: Half-precision (16-bit floats). Halves storage footprint and VRAM with practically identical detection accuracy.
* **INT8**: 8-bit integer quantization for maximum throughput on edge devices and embedded processors.

In [ ]:
# Export FP16 Half-Precision model
print("--- Exporting FP16 Half-Precision Model ---")
onnx_fp16_path = model.export(format="onnx", half=True, simplify=True)
print(f"Exported FP16 ONNX: {onnx_fp16_path}")

# Compare Storage Footprint
fp32_size_mb = os.path.getsize(str(pt_model_path)) / (1024 * 1024)
fp16_size_mb = os.path.getsize(str(onnx_fp16_path)) / (1024 * 1024)
savings_pct = ((fp32_size_mb - fp16_size_mb) / fp32_size_mb) * 100

print("\n--- Storage Footprint Comparison ---")
print(f"PyTorch FP32 Size:   {fp32_size_mb:.2f} MB")
print(f"ONNX FP16 Size:      {fp16_size_mb:.2f} MB")
print(f"Disk Space Saved:    {savings_pct:.1f}%")


---
## Step 4: Store on High-Speed Local Storage & Benchmark Cold-Starts

Measure the disk read time when loading model weights from disk to RAM. Keeping models on high-speed NVMe SSDs minimizes service boot times and worker recovery.

In [ ]:
# Measure cold-start load duration from local storage
start_time = time.perf_counter()
bench_model = YOLO(str(pt_model_path))
load_elapsed_ms = (time.perf_counter() - start_time) * 1000

print(f"Model path: {pt_model_path}")
print(f"Model disk load latency: {load_elapsed_ms:.2f} ms")
del bench_model


---
## Step 5: Production Singleton Pattern & Warm-Up Pass

> **Critical Production Rule:** Never re-instantiate `YOLO(...)` inside your frame loop or endpoint handler.
> Implement a **Thread-Safe Singleton** with an automated **GPU Warm-Up Pass**.

In [ ]:
import threading

class ModelSingleton:
    """Thread-safe singleton model manager with automated CUDA warm-up."""
    _instance = None
    _lock = threading.Lock()

    def __new__(cls, model_path: str = "yolov8n.pt", device: str = "auto"):
        with cls._lock:
            if cls._instance is None:
                cls._instance = super(ModelSingleton, cls).__new__(cls)
                print(f"[Singleton] Loading weights from: {model_path}...")
                cls._instance.model = YOLO(model_path)
                
                # Resolve device
                if device == "auto":
                    cls._instance.device = "cuda:0" if torch.cuda.is_available() else "cpu"
                else:
                    cls._instance.device = device
                
                if cls._instance.device != "cpu":
                    cls._instance.model.to(cls._instance.device)
                
                print(f"[Singleton] Model placed on compute device: {cls._instance.device}")
                cls._instance._warmup()
        return cls._instance

    def _warmup(self):
        """Runs dummy forward passes to compile CUDA kernels and prevent first-frame lag."""
        print("[Singleton] Running warm-up inference (3 cycles)...", end=" ")
        dummy_frame = np.zeros((640, 640, 3), dtype=np.uint8)
        for _ in range(3):
            _ = self.model.predict(dummy_frame, verbose=False, device=self.device)
        print("Done! Model is warmed up and ready.")

    def predict(self, frame, **kwargs):
        return self.model.predict(frame, device=self.device, **kwargs)

# Instantiate the singleton
model_manager = ModelSingleton(str(pt_model_path), device="auto")

# Calling it again returns the same in-memory instance without disk reads
model_manager_2 = ModelSingleton(str(pt_model_path))
print(f"Are both references pointing to the exact same instance? {model_manager is model_manager_2}")


---
## Step 6: Live Inference Latency & FPS Benchmark

Simulate processing a stream of video frames using our optimized singleton setup.

In [ ]:
# Generate synthetic 1080p video frame
test_frame = np.random.randint(0, 255, (1080, 1920, 3), dtype=np.uint8)

num_frames = 25
latencies = []

print(f"Simulating stream of {num_frames} frames through singleton model...")
for frame_idx in range(num_frames):
    start_t = time.perf_counter()
    _ = model_manager.predict(test_frame, imgsz=640, verbose=False)
    latency_ms = (time.perf_counter() - start_t) * 1000
    latencies.append(latency_ms)

avg_latency = float(np.mean(latencies))
min_latency = float(np.min(latencies))
max_latency = float(np.max(latencies))
fps = 1000.0 / avg_latency if avg_latency > 0 else 0

print("\n--- Benchmark Results ---")
print(f"Average Latency: {avg_latency:.2f} ms")
print(f"Min / Max:       {min_latency:.2f} ms / {max_latency:.2f} ms")
print(f"Throughput:      {fps:.1f} FPS")


---
### Production Deployment Checklist

| Optimization Principle | Recommended Action | Outcome |
| :--- | :--- | :--- |
| **1. Runtime Format** | Export to `.onnx` or TensorRT (`.engine`) | Reduces framework overhead and fuses layers |
| **2. Quantization** | Export with `half=True` (FP16) or INT8 | 50% smaller weights, 2x faster GPU throughput |
| **3. Fast Storage** | Host on local NVMe SSD | Minimizes cold start times and recovery delay |
| **4. Git Hygiene** | Keep weight directories in `.gitignore` | Prevents repo bloat and allows clean CI/CD |
| **5. Memory Management** | Load once via Singleton & warm-up dummy pass | Eliminates first-frame lag and per-request latency |